Import the working directories that I will be using

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path('/home/samb/projects/foia-scraper')))
from src.config import PROJECT_ROOT, DOWNLOAD_DIR, METADATA_FILE
TEST_PDF_DIR = PROJECT_ROOT/'notebooks/test_pdfs'
TEST_TXT_DIR = PROJECT_ROOT/'notebooks/test_txt'

I'm going to manually select pdfs for testing extraction methods, so that I can get one's that fulfil specific criteria. Examples and Justifications below:
0) foi-25-0029-ld-living-my-way-acquittals.pdf
    - This document contains no extractable text aside from the FOIA watermark
1) foi-25-0134-ld-minister-wells-diary.pdf
    - This document contains no extractable text aside from the FOIA watermark
    - This document has an interesting layout as it is a diary w/ redactions 
2) foi-25-0255-ld-mo-butler-speech-6-feb_0.pdf
    - Mix of tables and 'normal' text. Mostly non-extractable
3) foi-25-0211-ld-document-relating-to-the-future-fit-study-part-of-foi-25-0077-ld.pdf
    - This document is an email trail, and contains a large possibility of returning 'interesting' values
4) foi-25-0299-ld-baby-formula-products-at-wha-2025.pdf
    - This document contains massive redections
5) foi-25-0288-ld-data-related-to-hcps-and-chsp.pdf
    - This document also contains no extractable text aside from watermark
    - This document is primarily tables
6) foi-25-0431-ld-indigenous-liaison-officer-role.pdf
    - This document contains a mix of extractable text and non-extractable text
7) foi-25-0466-ld-costing-for-cdm-framework-changes-2023-24-budget.pdf
    - This document contains massive reductions, small text, and tables
8) foi-25-0458-letter-from-western-nsw-phn-ceo-appointment.pdf
    - This document is almost entirely redacted
9) foi-26-1951-access-to-official-ses-diary-entries.pdf
    - This document is just screenshots of MS Team/Outlook calendars, unusual format, no extractable text
10) foi-25-0322-ld-mo-correspondence-with-the-saturday-paper.pdf
    - Best case scenario, large amounts of extractable text, mostly non-redacted

In [ ]:
#import pdfplumber
import pymupdf
import json
import os
#os.listdir(TEST_PDF_DIR)

BEST_CASE = TEST_PDF_DIR/'foi-25-0322-ld-mo-correspondence-with-the-saturday-paper.pdf'

for pdf in os.listdir(TEST_PDF_DIR):
    txt_file_name = pdf[:-4]
    document = pymupdf.open(TEST_PDF_DIR/pdf)
    contents = ''
    for page in document:
        contents += page.get_text()
    with open(f'{TEST_TXT_DIR}/{txt_file_name}.txt', 'w') as f:
        print(contents, file=f)

pdfplumber doesn't work well with watermarks. PyMuPDF is the way to go
Watermarks will be left in as an artifact. Not worth the time to try and extract. Plus any method may also remove the s##x tag I'm interested in analysing as well

In [4]:
#OCR
import easyocr
import pytesseract
from pdf2image import convert_from_path
import time
import numpy as np
#
TEST_CASE =  TEST_PDF_DIR/'foi-25-0029-ld-living-my-way-acquittals.pdf'
#print("Converting PDF...")
reader = easyocr.Reader(['en'], gpu=True)
dpi_list = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
for dpi in dpi_list:
    start_total = time.time()
    pages = convert_from_path(TEST_CASE, dpi)
    
    pytesseract_text = ''
    easyocr_text = ''
    
    start_pt = time.time()
    for i, page in enumerate(pages):
        pytesseract_text += pytesseract.image_to_string(page) + '\n'
    pt_time = time.time() - start_pt
    
    start_eo = time.time()
    for i, page in enumerate(pages):
        result = reader.readtext(np.array(page), detail=0)
        easyocr_text += ' '.join(result) + '\n'
    eo_time = time.time() - start_eo
    
    print(f"DPI={dpi} | pytesseract: {pt_time:.1f}s | easyocr: {eo_time:.1f}s | total: {time.time()-start_total:.1f}s")
    with open(f"{TEST_TXT_DIR}/OCR_TEST (pytesseract, dpi={dpi}).txt", 'w') as f:
        print(pytesseract_text, file=f)
    with open(f"{TEST_TXT_DIR}/OCR_TEST (easyocr, dpi={dpi}).txt", 'w') as f:
        print(easyocr_text, file=f)


DPI=100 | pytesseract: 1.5s | easyocr: 3.5s | total: 5.2s
DPI=200 | pytesseract: 3.7s | easyocr: 6.4s | total: 10.6s
DPI=300 | pytesseract: 5.2s | easyocr: 7.7s | total: 13.6s
DPI=400 | pytesseract: 6.6s | easyocr: 8.1s | total: 16.0s
DPI=500 | pytesseract: 8.9s | easyocr: 8.1s | total: 18.8s
DPI=600 | pytesseract: 12.0s | easyocr: 8.7s | total: 23.3s
DPI=700 | pytesseract: 16.2s | easyocr: 8.7s | total: 28.5s
DPI=800 | pytesseract: 20.5s | easyocr: 9.1s | total: 34.2s
DPI=900 | pytesseract: 26.7s | easyocr: 8.4s | total: 40.9s


/home/samb/projects/foia-scraper/venv/lib64/python3.14/site-packages/PIL/Image.py:3451: DecompressionBombWarning: Image size (96682565 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


DPI=1000 | pytesseract: 31.3s | easyocr: 8.5s | total: 46.6s


In [36]:
#OCR
import easyocr
from pdf2image import convert_from_path
import time
import numpy as np
#
TEST_CASE =  TEST_PDF_DIR/'foi-25-0029-ld-living-my-way-acquittals.pdf'
#TEST_CASE = TEST_PDF_DIR/'foi-25-0322-ld-mo-correspondence-with-the-saturday-paper.pdf'
#print("Converting PDF...")
reader = easyocr.Reader(['en'], gpu=True)
dpi_list = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
for dpi in range(0, 700, 100):
    start_total = time.time()
    pages = convert_from_path(TEST_CASE, dpi)
    easyocr_text = ''
    start_eo = time.time()
    for i, page in enumerate(pages):
        result = reader.readtext(np.array(page), detail=0)
        easyocr_text += ' '.join(result) + '\n'
    eo_time = time.time() - start_eo
    
    print(f"DPI={dpi} | easyocr: {eo_time:.1f}s | total: {time.time()-start_total:.1f}s")
    with open(f"{TEST_TXT_DIR}/OCR_TEST v2 (easyocr, dpi={dpi}).txt", 'w') as f:
        print(easyocr_text, file=f)


DPI=0 | easyocr: 4.5s | total: 4.8s
DPI=100 | easyocr: 3.2s | total: 3.3s
DPI=200 | easyocr: 7.0s | total: 7.4s
DPI=300 | easyocr: 7.9s | total: 8.6s
DPI=400 | easyocr: 8.3s | total: 9.5s
DPI=500 | easyocr: 8.5s | total: 10.2s
DPI=600 | easyocr: 8.2s | total: 10.9s


In [6]:
import os

txt_dir = TEST_TXT_DIR  # or wherever they're saved

files = sorted([f for f in os.listdir(txt_dir) if 'v2' in f and 'easyocr' in f])
for filename in files:
    with open(txt_dir / filename, 'r') as f:
        text = f.read()
        word_count = len(text.split())
        print(f"{filename}: {word_count} words")

OCR_TEST v2 (easyocr, dpi=0).txt: 565 words
OCR_TEST v2 (easyocr, dpi=100).txt: 521 words
OCR_TEST v2 (easyocr, dpi=1000).txt: 567 words
OCR_TEST v2 (easyocr, dpi=1100).txt: 566 words
OCR_TEST v2 (easyocr, dpi=1200).txt: 566 words
OCR_TEST v2 (easyocr, dpi=1300).txt: 567 words
OCR_TEST v2 (easyocr, dpi=200).txt: 570 words
OCR_TEST v2 (easyocr, dpi=300).txt: 567 words
OCR_TEST v2 (easyocr, dpi=400).txt: 569 words
OCR_TEST v2 (easyocr, dpi=500).txt: 567 words
OCR_TEST v2 (easyocr, dpi=600).txt: 567 words
OCR_TEST v2 (easyocr, dpi=700).txt: 567 words
OCR_TEST v2 (easyocr, dpi=800).txt: 567 words
OCR_TEST v2 (easyocr, dpi=900).txt: 569 words


In [73]:
import pymupdf
import os
import re

# ============================================================
# WATERMARK COLOR CHECK - all test PDFs
# Outputs span color, flags, size and text for each PDF
# Used to verify watermark color consistency across test set
# ============================================================

output_file = TEST_TXT_DIR / 'watermark_color_check.txt'
pattern = r"FOI \d{2}-\d{4} [A-Za-z]{2} - Document \d+"


with open(output_file, 'w') as f:
    for pdf in os.listdir(TEST_PDF_DIR):
        f.write(f"\n=== {pdf} ===\n")
        document = pymupdf.open(TEST_PDF_DIR / pdf)
        for page in document:
            blocks = page.get_text("dict")["blocks"]
            for block in blocks:
                if block['type'] == 0:
                    for line in block['lines']:
                        for span in line['spans']:
                            #if abs(line['dir'][0] - 0.7071) < 0.001:
                            #if 'freedom of information act' in span['text'].lower():
                            if re.match(pattern.lower(), span['text'].lower()):
                                f.write(f"alpha={span['alpha']} color={span['color']} dir={line['dir']} {repr(span['text'])}\n")

print(f"Output saved to {output_file}")

Output saved to /home/samb/projects/foia-scraper/notebooks/test_txt/watermark_color_check.txt


## Phase 2: Text Extraction

### Approach
PDFs in this dataset fall into two categories:
- **Native text PDFs** — digitally created, text can be extracted directly using pymupdf
- **Scanned PDFs** — image-based, require OCR (EasyOCR + pdf2image)

### Watermark Handling
Every document has a diagonal FOI watermark stamped across each page. This must be filtered out before extraction to avoid polluting the text data.

Analysis of the test set revealed the watermark can be reliably identified by the `dir` field in pymupdf's span dictionary — watermark text is always rendered at 45 degrees, represented as a unit vector `(0.7071..., -0.7071...)`. This is consistent across all 11 test documents regardless of alpha value or colour, which vary between documents.

A text content fallback (`'freedom of information act'`) is used as a secondary filter for robustness.

### OCR Method Selection
EasyOCR was selected over pytesseract for the following reasons:
- GPU-accelerated (RTX 2060), processing time largely DPI-independent above 300
- Cleaner output with less watermark noise bleed-through
- More consistent quality across DPI range

**Selected DPI: 300** — quality plateaus above this value for EasyOCR while conversion time continues to increase linearly.

In [89]:
pattern = r"FOI \d{2}-\d{4} [A-Za-z]{2} - Document \d+"
pattern = r"FOI\d{2}-\d{4}[A-Za-z]{2}-Document\d+"

pdf_file = 'foi-26-1951-access-to-official-ses-diary-entries.pdf'


with open(output_file, 'w') as f:
    for pdf in os.listdir(TEST_PDF_DIR):
        f.write(f"\n=== {pdf} ===\n")
        document = pymupdf.open(TEST_PDF_DIR / pdf_file)
        for page in document:
            blocks = page.get_text("dict")["blocks"]
            for block in blocks:
                    if block['type'] == 0:
                        block_text = ''
                        for line in block['lines']:
                            for span in line['spans']:
                            #if int(abs(line['dir'][0]) * 10_000) == 7071 and int(abs(line['dir'][1]) * 10_000) == 7071:
                                cleaned_text = re.sub(r'\s+', '', str(span['text']))
                                if (
                                    int(abs(line['dir'][0]) * 10_000) != 7071 and
                                    int(abs(line['dir'][1]) * 10_000) != 7071 and
                                    not re.match(pattern.strip(), cleaned_text, re.IGNORECASE)
                                 ):
                                #f.write(f"alpha={span['alpha']} color={span['color']} dir={line['dir']} {repr(span['text'])}\n")
                                    block_text += span['text']
                                print(f"Original: {repr(span['text'])}")
                                print(f"Cleaned: {repr(cleaned_text)}")
                        if block_text.strip():
                            f.write(block_text + '\n')

print(f"Output saved to {output_file}")

Original: 'FOI 26-1951 - Document 1 '
Cleaned: 'FOI26-1951-Document1'
Original: 'Page 1 of 1'
Cleaned: 'Page1of1'
Original: 'This document has been released under '
Cleaned: 'Thisdocumenthasbeenreleasedunder'
Original: 'the Freedom Of Information Act 1982 by'
Cleaned: 'theFreedomOfInformationAct1982by'
Original: 'the Department of Health, Disability and Ageing '
Cleaned: 'theDepartmentofHealth,DisabilityandAgeing'
Original: 'This document has been released under '
Cleaned: 'Thisdocumenthasbeenreleasedunder'
Original: 'the Freedom Of Information Act 1982 by  '
Cleaned: 'theFreedomOfInformationAct1982by'
Original: 'the Department of Health, Disability and Ageing '
Cleaned: 'theDepartmentofHealth,DisabilityandAgeing'
Original: ' '
Cleaned: ''
Original: 'This document has been released under '
Cleaned: 'Thisdocumenthasbeenreleasedunder'
Original: 'the Freedom Of Information Act 1982 by  '
Cleaned: 'theFreedomOfInformationAct1982by'
Original: 'the Department of Health, Disability and Ageing

In [37]:
#OCR
import easyocr
import time
import pymupdf
import numpy as np

TEST_CASE = TEST_PDF_DIR / 'foi-25-0029-ld-living-my-way-acquittals.pdf'
reader = easyocr.Reader(['en'], gpu=True)

for dpi in range(100, 800, 100):
    start_total = time.time()
    easyocr_text = ''
    document = pymupdf.open(TEST_CASE)
    
    start_eo = time.time()
    for page in document:
        # Redact watermark
        for block in page.get_text("dict")["blocks"]:
            if block['type'] == 0:
                for line in block['lines']:
                    if int(abs(line['dir'][0]) * 10_000) == 7071:
                        for span in line['spans']:
                            page.delete_text(span['bbox'])
        page.apply_redactions()
        
        # Render to image
        pix = page.get_pixmap(matrix=pymupdf.Matrix(dpi/72, dpi/72))
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        print(f"width={pix.width} height={pix.height} n={pix.n} samples_len={len(pix.samples)}")
        result = reader.readtext(img[:,:,:3], detail=0)
        easyocr_text += ' '.join(result) + '\n'
    eo_time = time.time() - start_eo
    
    print(f"DPI={dpi} | easyocr: {eo_time:.1f}s | total: {time.time()-start_total:.1f}s")
    with open(f"{TEST_TXT_DIR}/OCR_TEST v3 (easyocr, dpi={dpi}).txt", 'w') as f:
        print(easyocr_text, file=f)

AttributeError: 'Page' object has no attribute 'delete_text'

In [43]:
import easyocr
import time
import pymupdf
import numpy as np
import cv2

TEST_CASE = TEST_PDF_DIR / 'foi-25-0029-ld-living-my-way-acquittals.pdf'
reader = easyocr.Reader(['en'], gpu=True)

for dpi in range(100, 800, 100):
    start_total = time.time()
    easyocr_text = ''
    document = pymupdf.open(TEST_CASE)
    
    start_eo = time.time()
    for page in document:
        # Collect watermark span bboxes
        watermark_bboxes = []
        for block in page.get_text("dict")["blocks"]:
            if block['type'] == 0:
                for line in block['lines']:
                    if int(abs(line['dir'][0]) * 10_000) == 7071:
                        for span in line['spans']:
                            watermark_bboxes.append(span['bbox'])
        
        # Render to image
        pix = page.get_pixmap(matrix=pymupdf.Matrix(dpi/72, dpi/72))
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n).copy()
        
        # Paint over individual span bboxes with white
        scale = dpi / 72
        for bbox in watermark_bboxes:
            x0, y0, x1, y1 = [int(c * scale) for c in bbox]
            cv2.rectangle(img, (x0, y0), (x1, y1), (255, 255, 255), -1)
        
        result = reader.readtext(img, detail=0)
        easyocr_text += ' '.join(result) + '\n'
    
    eo_time = time.time() - start_eo
    print(f"DPI={dpi} | easyocr: {eo_time:.1f}s | total: {time.time()-start_total:.1f}s")
    
    with open(f"{TEST_TXT_DIR}/OCR_TEST_v4_dpi_{dpi}.txt", 'w') as f:
        print(easyocr_text, file=f)

DPI=100 | easyocr: 1.7s | total: 1.7s
DPI=200 | easyocr: 4.1s | total: 4.1s
DPI=300 | easyocr: 4.7s | total: 4.7s
DPI=400 | easyocr: 5.3s | total: 5.3s
DPI=500 | easyocr: 5.5s | total: 5.5s
DPI=600 | easyocr: 5.6s | total: 5.6s
DPI=700 | easyocr: 6.2s | total: 6.2s


In [18]:
import easyocr
import time
import pymupdf
import numpy as np

TEST_CASE = TEST_PDF_DIR / 'foi-25-0029-ld-living-my-way-acquittals.pdf'
reader = easyocr.Reader(['en'], gpu=True)

for dpi in range(100, 800, 100):
    start_total = time.time()
    easyocr_text = ''
    document = pymupdf.open(TEST_CASE)
    
    # Toggle off all watermark OCG layers
    document.set_layer_ui_config('Watermark', action=2) 
    
    start_eo = time.time()
    for page in document:
        pix = page.get_pixmap(matrix=pymupdf.Matrix(dpi/72, dpi/72))
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n).copy()
        
        result = reader.readtext(img, detail=0)
        easyocr_text += ' '.join(result) + '\n'
    
    eo_time = time.time() - start_eo
    print(f"DPI={dpi} | easyocr: {eo_time:.1f}s | total: {time.time()-start_total:.1f}s")
    
    with open(f"{TEST_TXT_DIR}/OCR_TEST_v5_dpi_{dpi}.txt", 'w') as f:
        print(easyocr_text, file=f)

ValueError: bad OCG 'Watermark'.

In [19]:
import easyocr
import time
import pymupdf
import numpy as np
import os

TEST_CASE = TEST_PDF_DIR / 'foi-25-0029-ld-living-my-way-acquittals.pdf'
reader = easyocr.Reader(['en'], gpu=True)

for dpi in range(100, 800, 100):
    start_total = time.time()
    easyocr_text = ''
    
    document = pymupdf.open(TEST_CASE)
    cat = document.pdf_catalog()
    
    # Dynamically find and disable watermark OCG
    watermark_xrefs = [xref for xref, props in document.get_ocgs().items() if props['name'] == 'Watermark']
    if watermark_xrefs:
        off_array = ' '.join(f"{xref} 0 R" for xref in watermark_xrefs)
        document.xref_set_key(cat, "OCProperties/D/ON", "[]")
        document.xref_set_key(cat, "OCProperties/D/OFF", f"[{off_array}]")
    
    # Save and reload to apply OCG changes
    tmp_path = "/tmp/tmp_no_watermark.pdf"
    document.save(tmp_path)
    document = pymupdf.open(tmp_path)
    
    start_eo = time.time()
    for page in document:
        pix = page.get_pixmap(matrix=pymupdf.Matrix(dpi/72, dpi/72))
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n).copy()
        result = reader.readtext(img, detail=0)
        easyocr_text += ' '.join(result) + '\n'
    
    eo_time = time.time() - start_eo
    print(f"DPI={dpi} | easyocr: {eo_time:.1f}s | total: {time.time()-start_total:.1f}s")
    
    with open(f"{TEST_TXT_DIR}/OCR_TEST_v5_dpi_{dpi}.txt", 'w') as f:
        print(easyocr_text, file=f)

os.remove(tmp_path)

DPI=100 | easyocr: 3.5s | total: 3.5s
DPI=200 | easyocr: 6.7s | total: 6.7s
DPI=300 | easyocr: 7.8s | total: 7.8s
DPI=400 | easyocr: 8.3s | total: 8.3s
DPI=500 | easyocr: 8.5s | total: 8.5s
DPI=600 | easyocr: 9.6s | total: 9.6s
DPI=700 | easyocr: 10.8s | total: 10.8s


In [28]:
import sqlite3

conn = sqlite3.connect('text_extract_test.db')
curs = conn.cursor()
curs.execute('DROP TABLE IF EXISTS TEXT_STORAGE')
curs.execute('DROP TABLE IF EXISTS DOCUMENTS')
curs.execute('DROP TABLE IF EXISTS PAGES')
create_table_documents = """
                CREATE TABLE DOCUMENTS (
                    FILENAME,
                    FOI_REFERENCE_NUMBER,
                    PAGE_COUNT,
                    DOCUMENT_WORD_COUNT
                );
                """
create_table_pages = """
                CREATE TABLE PAGES (
                    FILENAME,
                    FOI_REFERENCE_NUMBER,
                    PYTHON_PAGE_NUMBER,
                    HUMAN_PAGE_NUMBER,
                    RAW_TEXT,
                    WORD_COUNT,
                    DATE_PROCESSED,
                    WATERMARK_REMOVED,
                    EXTRACTION_METHOD
                );
                """
curs.execute(create_table_documents)
curs.execute(create_table_pages)
conn.commit()
conn.close()

In [ ]:
import pymupdf
import os
import sqlite3
import time
import re
import string

VALID_ALNUM = set(string.ascii_letters + string.digits)

foi_watermark_pattern = r"FOI\d+-\d+(?:\w+)?(?:-)?(?:DOCUMENT\d*)?"
pagenumber_watermark_pattern = r"PAGE\d+OF\d+"
ligature_list = []

def remove_watermarks (s) :
    #remove whitespace
    s = re.sub(r"\s+", "", s, flags=re.IGNORECASE)
    #remove known watermark patters
    s = re.sub(foi_watermark_pattern, "", s, flags=re.IGNORECASE)
    s = re.sub(pagenumber_watermark_pattern, "", s, flags=re.IGNORECASE)
    return s



conn = sqlite3.connect('text_extract_test.db')
curs = conn.cursor()
for pdf in os.listdir(TEST_PDF_DIR):
    if not pdf.lower().endswith(".pdf"):
        continue
    
    #DATE PROCESSEDprocessing_time_start = time.time()
    processing_time_start = time.time()
    date_processed = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(processing_time_start))
    newly_processed_pages = 0

    try:
        filename = pdf
        foi_reference_number = pdf[:-4]
        document = pymupdf.open(TEST_PDF_DIR/pdf)
        #TOTAL PAGES
        total_pages = len(document)
        #EXTRACTION METHOD
        extraction_method = 'NATIVE'
        #WATERMARK
        watermark = 'N/A'
        document_word_count = 0
        doc_exists = curs.execute("SELECT FILENAME, FOI_REFERENCE_NUMBER, PAGE_COUNT, DOCUMENT_WORD_COUNT FROM DOCUMENTS WHERE FILENAME = ?", (filename,)).fetchall()
        if doc_exists:
            document_word_count = doc_exists[0][3]
            print(document_word_count)

        curs.execute("SELECT PYTHON_PAGE_NUMBER FROM PAGES WHERE FILENAME = ?", (filename,))
        processed_pages = {row[0] for row in curs.fetchall()}
        

        for i, page in enumerate(document):
            if i in processed_pages:
                continue
            #PAGE NUMBER
            python_page_number = i
            human_page_number = i+1

            #RAW TEXT
            contents = ''
            blocks = page.get_text("dict")["blocks"]
            for block in blocks:
                if block['type'] == 0:
                    for line in block['lines']:
                        if int(abs(line['dir'][0]) * 10_000) != 7071:  # skip watermark
                            for span in line['spans']:
                                contents += span['text']
                            contents += '\n'
                    contents += '\n'
            #WORD COUNT
            if len(remove_watermarks(contents)) == 0:
                contents = None
                word_count = 0
            else:
                word_count = len(contents.split())
            document_word_count += word_count
            if contents is None:
                continue
            else:
                curs.execute("""INSERT INTO PAGES (FILENAME, FOI_REFERENCE_NUMBER, PYTHON_PAGE_NUMBER, HUMAN_PAGE_NUMBER, RAW_TEXT, WORD_COUNT, DATE_PROCESSED, WATERMARK_REMOVED, EXTRACTION_METHOD) 
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""", (filename, foi_reference_number, python_page_number, human_page_number, contents, word_count, date_processed, watermark, extraction_method)
                        )
                newly_processed_pages += 1
        if doc_exists and newly_processed_pages > 0:
            curs.execute("""UPDATE DOCUMENTS SET PAGE
            
            """)
        if newly_processed_pages > 0:
            curs.execute("""INSERT INTO DOCUMENTS (FILENAME, FOI_REFERENCE_NUMBER, PAGE_COUNT, DOCUMENT_WORD_COUNT) 
                            VALUES (?, ?, ?, ?)""", (filename, foi_reference_number, total_pages, document_word_count)
                        )
        conn.commit()
    except Exception as e:
        print(f"Error processing {pdf}: {e}")
        with open('errors.txt', 'a') as f:
            print(f"{date_processed} - Error Processing ({pdf}): {e}", file=f) 
        continue
conn.close()

5006
767
726
3414
1761
2
3419
1453


In [8]:
#conn.close()
print('Ɵ'.isalpha())

True


In [ ]:

import sqlite3
import pandas as pd

conn = sqlite3.connect('text_extract_test.db')
curs = conn.cursor()
query = """
    SELECT
          d.FILENAME
        , p.PYTHON_PAGE_NUMBER
        , p.HUMAN_PAGE_NUMBER
        , p.RAW_TEXT
        , p.WORD_COUNT
    FROM DOCUMENTS d
    LEFT JOIN PAGES p    on d.FILENAME = p.FILENAME
                        and d.FOI_REFERENCE_NUMBER = p.FOI_REFERENCE_NUMBER
    WHERE 1=1
    AND RAW_TEXT IS NULL;
        """
df = pd.read_sql_query(query, conn)
conn.close()
unprocessed = df.to_dict()
print(unprocessed)


{'FILENAME': {0: 'foi-25-0134-ld-minister-wells-diary.pdf', 1: 'foi-25-0134-ld-minister-wells-diary.pdf', 2: 'foi-25-0134-ld-minister-wells-diary.pdf', 3: 'foi-25-0134-ld-minister-wells-diary.pdf', 4: 'foi-25-0134-ld-minister-wells-diary.pdf', 5: 'foi-25-0134-ld-minister-wells-diary.pdf', 6: 'foi-25-0134-ld-minister-wells-diary.pdf', 7: 'foi-25-0134-ld-minister-wells-diary.pdf', 8: 'foi-25-0299-ld-baby-formula-products-at-wha-2025.pdf', 9: 'foi-25-0299-ld-baby-formula-products-at-wha-2025.pdf', 10: 'foi-25-0299-ld-baby-formula-products-at-wha-2025.pdf', 11: 'foi-25-0299-ld-baby-formula-products-at-wha-2025.pdf', 12: 'foi-25-0299-ld-baby-formula-products-at-wha-2025.pdf', 13: 'foi-25-0299-ld-baby-formula-products-at-wha-2025.pdf', 14: 'foi-25-0299-ld-baby-formula-products-at-wha-2025.pdf', 15: 'foi-25-0299-ld-baby-formula-products-at-wha-2025.pdf', 16: 'foi-25-0288-ld-data-related-to-hcps-and-chsp.pdf', 17: 'foi-25-0288-ld-data-related-to-hcps-and-chsp.pdf', 18: 'foi-25-0288-ld-data-rel